In [ ]:
%pip install -e ../../../

In [ ]:
import pandas as pd

COLUMNS_OPTIONS = [
    'option_id',
    'name',
    'flow_rate-nominal',
    'lifetime-min',
    'lifetime-max'
]

COLUMNS_CURVES = [
    'flow_rate',
    'head',
    'break_power',
    'efficiency'
]

PUMPS_DATA = {
    "065-040-200 Φ 184": [
        (10, 45,   3.7, 33.14),
        (20, 44,   4.5, 53.29),
        (30, 41,   5.5, 60.94),
        (40, 37.5, 6.5, 62.88),
        (50, 30,   7.5, 54.50),
    ],
    "080-065-200 Φ175": [
        (20, 41, 5.0, 44.69),
        (40, 40.5, 7.0, 63.84),
        (60, 40, 8.5, 76.94),
        (80, 37, 10.5, 76.82),
        (100, 33, 12.0, 74.94),
    ],
    "100-080-200 Φ180": [
        (50, 44, 10.25, 58.49),
        (85, 41, 15.0, 63.31),
        (120, 40, 17.5, 74.74),
        (155, 39, 20.0, 82.36),
        (200, 31, 22.5, 75.09),
    ],
    "150-125-200": [
        (100, 55, 42.0, 35.68),
        (200, 54, 50.0, 58.86),
        (300, 51, 58.0, 71.88),
        (400, 47, 61.0, 83.98),
        (500, 41, 62.7, 89.09),
    ]
}

LIFETIME_MIN = 25
LIFETIME_MAX = 30

def option_id_gen(x: int) -> str:
    return f"PU{x:02d}"

pump_options_df = pd.DataFrame(columns=COLUMNS_OPTIONS)

curves_dfs = {}
for name, curve_data in PUMPS_DATA.items():
    pump_id = option_id_gen(len(pump_options_df)+1)

    curve_df = pd.DataFrame(data=curve_data, columns=COLUMNS_CURVES)
    curves_dfs[pump_id] = curve_df
    
    nominal_flow_rate = curve_df.set_index('flow_rate')['efficiency'].idxmax()

    pump_options_df.loc[len(pump_options_df)] = [
        pump_id,
        name,
        nominal_flow_rate,
        LIFETIME_MIN,
        LIFETIME_MAX
    ]

In [ ]:
from water_futures_battle.core.base_model import StaticProperties

pump_options_sps = StaticProperties('pump_options-static_properties')
pump_options_sps['options'] = pump_options_df.copy()
for pump_id, curve_df in curves_dfs.items():
    pump_options_sps[pump_id] = curve_df

pump_options_sps.dump(path='..')